# 01 数据理解

## 本文件目标
理解 Kaggle Store Sales 项目的数据结构，确认哪些表可以安全合并，建立最初的数据底座。

## 本文件主要工作
1. 读取 train / test / stores / oil / holidays / transactions / sample_submission
2. 查看每张表的 shape、head、dtype、missing
3. 判断各表作用
4. 将 stores 和 oil 合并到 train / test
5. 对油价缺失值进行简单填充

## 当前结论
- 可以先安全合并：stores、oil
- 暂不直接合并：holidays、transactions
- 这是时间序列问题，后续验证不能使用随机 KFold

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train = pd.read_csv(PATH + "train.csv", parse_dates=["date"])
test = pd.read_csv(PATH + "test.csv", parse_dates=["date"])
stores = pd.read_csv(PATH + "stores.csv")
oil = pd.read_csv(PATH + "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(PATH + "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(PATH + "transactions.csv", parse_dates=["date"])
sample_submission = pd.read_csv(PATH + "sample_submission.csv")

In [ ]:
def quick_look(df, name):
    print(f"\n========== {name} ==========")
    print("shape:", df.shape)
    print("\nhead:")
    print(df.head())
    print("\ndtypes:")
    print(df.dtypes)
    print("\nmissing:")
    print(df.isna().sum().sort_values(ascending=False).head(10))

quick_look(train, "train")
quick_look(test, "test")
quick_look(stores, "stores")
quick_look(oil, "oil")
quick_look(holidays, "holidays")
quick_look(transactions, "transactions")
quick_look(sample_submission, "sample_submission")

## 为什么先合并 stores 和 oil
- `stores.csv` 按 `store_nbr` 一店一行，结构稳定，不容易炸表。
- `oil.csv` 按 `date` 提供油价信息，适合作为日期级外部特征。

## 为什么暂不直接合并 holidays 和 transactions
- `holidays_events.csv` 同一天可能有多条记录，直接按 `date` merge 会炸表。
- `transactions.csv` 在测试期不天然可得，直接作为普通特征不够严谨。

In [ ]:
base_train = train.merge(stores, on="store_nbr", how="left")
base_test = test.merge(stores, on="store_nbr", how="left")

base_train = base_train.merge(oil, on="date", how="left")
base_test = base_test.merge(oil, on="date", how="left")

base_train = base_train.sort_values("date").copy()
base_test = base_test.sort_values("date").copy()

base_train["dcoilwtico"] = base_train["dcoilwtico"].ffill().bfill()
base_test["dcoilwtico"] = base_test["dcoilwtico"].ffill().bfill()

In [ ]:
print("base_train shape:", base_train.shape)
print("base_test shape:", base_test.shape)

print("\nbase_train missing:")
print(base_train.isna().sum().sort_values(ascending=False).head(10))

print("\nbase_test missing:")
print(base_test.isna().sum().sort_values(ascending=False).head(10))

print("\nbase_train dcoilwtico missing:", base_train["dcoilwtico"].isna().sum())
print("base_test dcoilwtico missing:", base_test["dcoilwtico"].isna().sum())

print("train date range:", train["date"].min(), "to", train["date"].max())
print("test date range:", test["date"].min(), "to", test["date"].max())

## 本阶段小结
Day 1 的重点不是训练模型，而是：
- 看懂表结构
- 建立安全的数据底座
- 确认后续应该按时间处理问题，而不是按普通 tabular 题处理